# Task 5 — Convolutional Neural Networks
**Course:** Machine Learning & Deep Learning  
**Points:** 10/60  
**School of Artificial Intelligence and Data Science**

---

## Overview
We build, train, and evaluate a Convolutional Neural Network (CNN) on an image classification task using:
- Custom CNN architecture with 3+ convolutional blocks
- Data augmentation (horizontal flip, random crop, color jitter)
- Learning rate scheduler
- Transfer learning with pre-trained models (ResNet-18)
- Visualization of learned filters, activation maps, and sample classifications


## Step 0 — Imports & Setup

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import ResNet50, VGG16, MobileNet
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix
import itertools

tf.random.set_seed(42)
np.random.seed(42)
sns.set_theme(style='whitegrid')

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

TensorFlow version: 2.21.0
GPU available: False


## Step 1 — Data Preparation

In [19]:
def prepare_cifar10_data():
    """Load CIFAR-10 dataset, normalize, and split into train/validation/test."""
    # Load CIFAR-10
    (x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()

    # Normalize pixel values to [0, 1]
    x_train_full = x_train_full.astype('float32') / 255.0
    x_test = x_test.astype('float32') / 255.0

    # Split training data into train and validation (90% train, 10% validation)
    x_val = x_train_full[-5000:]
    y_val = y_train_full[-5000:]
    x_train = x_train_full[:-5000]
    y_train = y_train_full[:-5000]

    # Convert class vectors to binary class matrices
    y_train = to_categorical(y_train, 10)
    y_val = to_categorical(y_val, 10)
    y_test = to_categorical(y_test, 10)

    print(f'Using CIFAR-10 dataset')
    print(f'Train: {x_train.shape}  Val: {x_val.shape}  Test: {x_test.shape}')
    print(f'Classes: {len(np.unique(np.argmax(y_train, axis=1)))}')
    
    return x_train, x_val, x_test, y_train, y_val, y_test

x_train, x_val, x_test, y_train, y_val, y_test = prepare_cifar10_data()
INPUT_SHAPE = x_train.shape[1:]  # (32, 32, 3)
NUM_CLASSES = y_train.shape[1]   # 10

Using CIFAR-10 dataset
Train: (45000, 32, 32, 3)  Val: (5000, 32, 32, 3)  Test: (10000, 32, 32, 3)
Classes: 10


## Step 2 — Custom CNN Architecture

### Architecture
```
Input (32x32x3) → Conv(32, 3x3) → BN → ReLU → MaxPool(2x2)
           → Conv(64, 3x3) → BN → ReLU → MaxPool(2x2)
           → Conv(128, 3x3) → BN → ReLU → MaxPool(2x2)
           → Flatten → Dense(256) → BN → ReLU → Dropout(0.5)
           → Dense(128) → BN → ReLU → Dropout(0.3)
           → Dense(10, Softmax)
```

In [20]:
def build_custom_cnn(input_shape, num_classes):
    """Build custom CNN with 3 convolutional blocks."""
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # First convolutional block
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),

        # Second convolutional block
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),

        # Third convolutional block
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),

        # Classification head
        layers.Flatten(),
        layers.Dense(256),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.5),
        layers.Dense(128),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ], name='custom_cnn')
    
    return model

# Build and show architecture
model_custom = build_custom_cnn(INPUT_SHAPE, NUM_CLASSES)
model_custom.summary()

Model: "custom_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_8 (Activation)       │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_9 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴─────────────

 Total params: 654,410 (2.50 MB)

 Trainable params: 653,194 (2.49 MB)

 Non-trainable params: 1,216 (4.75 KB)

In [30]:
# === Step 0 continued — Setup constants =========================
import os
import matplotlib
matplotlib.use('Agg')      # non-interactive backend (stable on headless / batch)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers, models, optimizers, callbacks

OUTPUT_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

sns.set_theme(style='darkgrid')
print(f"Outputs save to: {OUTPUT_DIR}")
print(f"Classes: {CLASS_NAMES}")


## Step 3 — Training with Data Augmentation


In [31]:
# Create data-augmentation generator
datagen = keras.preprocessing.image.ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
)
datagen.fit(x_train)

# Callbacks
lr_sched = callbacks.ReduceLROnPlateau(
    monitor='val_accuracy', factor=0.5, patience=3, verbose=1)
early_stop = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1)

# Compile
model_custom.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy', metrics=['accuracy'])

# Train (20 epochs, early stopping is active)
history_custom = model_custom.fit(
    x_train, y_train_cat,
    batch_size=128,
    validation_data=(x_val, y_val_cat),
    epochs=20,
    callbacks=[lr_sched, early_stop],
    verbose=2,
)


## Step 4 — Training History

In [32]:
# ─── Training History Plot ──────────────────────────────────────────
epochs_range = range(1, len(history_custom.history['accuracy']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history_custom.history['accuracy'],
             'b-', label='Train Accuracy')
axes[0].plot(epochs_range, history_custom.history['val_accuracy'],
             'g--', label='Val Accuracy')
axes[0].set_title('Custom CNN — Accuracy')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history_custom.history['loss'],
             'b-', label='Train Loss')
axes[1].plot(epochs_range, history_custom.history['val_loss'],
             'g--', label='Val Loss')
axes[1].set_title('Custom CNN — Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_training_history.png'), dpi=150)
plt.close()
print("Saved T5_training_history.png")


## Step 5 — Custom CNN Evaluation & Confusion Matrix

In [33]:
# ─── Evaluate Custom CNN ───────────────────────────────────────────
test_loss_c, test_acc_c = model_custom.evaluate(x_test, y_test_cat, verbose=0)
print(f"Custom CNN Test  — Accuracy: {test_acc_c:.4f}  Loss: {test_loss_c:.4f}")

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_pred_c = np.argmax(model_custom.predict(x_test, verbose=0), axis=1)
y_true = y_test.flatten()
cm_c = confusion_matrix(y_true, y_pred_c)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_c, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title(f'Custom CNN — Confusion Matrix (Accuracy: {test_acc_c:.4f})')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_custom_confusion_matrix.png'), dpi=150)
plt.close()
print("Saved T5_custom_confusion_matrix.png")

print("\nClassification Report:")
print(classification_report(y_true, y_pred_c, target_names=CLASS_NAMES))


## Step 6 — ResNet-50 Transfer Learning — Model Setup

In [34]:
# ─── ResNet-50 Transfer Learning (frozen backbone, trainable head) ───
from tensorflow.keras.applications import ResNet50

print("Loading ResNet-50 weights (ImageNet, no top)...")
base_resnet = ResNet50(weights='imagenet', include_top=False,
                       input_shape=(32, 32, 3))
base_resnet.trainable = False   # freeze backbone

model_rn = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
    base_resnet,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.BatchNormalization(),
    layers.Dense(10, activation='softmax')
], name='resnet50_transfer')

model_rn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy', metrics=['accuracy'])
model_rn.summary()


## Step 7 — ResNet-50 Training & History

In [35]:
# ─── Train ResNet-50 Transfer Model (frozen, 5 epochs) ─────────────
print("Training ResNet-50 (frozen backbone, 5 epochs)...")
hist_rn = model_rn.fit(
    x_train, y_train_cat,
    batch_size=64,
    validation_data=(x_val, y_val_cat),
    epochs=5,
    verbose=2,
)
# Save training history
epochs_rn = range(1, len(hist_rn.history['accuracy']) + 1)
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
axes2[0].plot(epochs_rn, hist_rn.history['accuracy'], 'r--', label='Train')
axes2[0].plot(epochs_rn, hist_rn.history['val_accuracy'], 'r-',  label='Val')
axes2[0].set_title('ResNet-50 Transfer — Accuracy')
axes2[0].legend(); axes2[0].grid(alpha=0.3)
axes2[1].plot(epochs_rn, hist_rn.history['loss'], 'r--', label='Train')
axes2[1].plot(epochs_rn, hist_rn.history['val_loss'], 'r-',  label='Val')
axes2[1].set_title('ResNet-50 Transfer — Loss')
axes2[1].legend(); axes2[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_resnet50_training_history.png'), dpi=150)
plt.close()
print("Saved T5_resnet50_training_history.png")


## Step 8 — ResNet-50 Evaluation & Confusion Matrix

In [36]:
# ─── Evaluate ResNet-50 on Test Set ─────────────────────────────────
test_loss_r, test_acc_r = model_rn.evaluate(x_test, y_test_cat, verbose=0)
print(f"ResNet-50 Test  — Accuracy: {test_acc_r:.4f}  Loss: {test_loss_r:.4f}")

from sklearn.metrics import classification_report, confusion_matrix

y_pred_r = np.argmax(model_rn.predict(x_test, verbose=0), axis=1)
cm_r = confusion_matrix(y_true, y_pred_r)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_r, annot=True, fmt='d', cmap='Purples',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title(f'ResNet-50 Transfer Learning — Confusion Matrix (Accuracy: {test_acc_r:.4f})')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_resnet50_confusion_matrix.png'), dpi=150)
plt.close()
print("Saved T5_resnet50_confusion_matrix.png")

print("\nResNet-50 Classification Report:")
print(classification_report(y_true, y_pred_r, target_names=CLASS_NAMES))


## Step 9 — Learned Filter Visualizations

In [37]:
# ─── First-Layer Convolutional Filter Weights ─────────────────────
first_conv = model_custom.layers[0]        # Conv2D, 32 filters
w = first_conv.get_weights()[0]            # shape (3, 3, 3, 32)

fig, axes = plt.subplots(4, 8, figsize=(22, 8))
axes = axes.flatten()
for i in range(32):
    f = w[:, :, 0, i]
    fmin, fmax = f.min(), f.max()
    f_norm = (f - fmin) / (fmax - fmin + 1e-8)
    axes[i].imshow(f_norm, cmap='viridis')
    axes[i].axis('off')
    axes[i].set_title(f'#{i}', fontsize=6)
plt.suptitle(
    'Custom CNN — First Conv Layer Learned Filters (R channel, 32/32 shown)',
    fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_cnn_filters.png'), dpi=150)
plt.close()
print("Saved T5_cnn_filters.png")


## Step 10 — Activation Map Visualizations

In [38]:
# ─── Activation Maps — Conv Block 1 ─────────────────────────────────
sample_img = x_test[0:1]   # shape (1, 32, 32, 3)

# Build a model that returns intermediate Conv2D + Activation layer outputs
conv_outputs = [l.output for l in model_custom.layers
                if isinstance(l, (layers.Conv2D, layers.Activation))]
act_model = keras.Model(inputs=model_custom.inputs[0], outputs=conv_outputs)

activations = act_model.predict(sample_img, verbose=0)

# activations[4] = output of the first conv block (1, 16, 16, 32)
acts_c1 = activations[4]

fig, axes = plt.subplots(2, 8, figsize=(22, 5))
axes = axes.flatten()
for i in range(min(16, acts_c1.shape[-1])):
    axes[i].imshow(acts_c1[0, :, :, i], cmap='viridis')
    axes[i].axis('off')
    axes[i].set_title(f'Ch {i}', fontsize=7)

plt.suptitle(
    'CNN Activation Maps — Conv Block 1 (first test image)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_cnn_activations.png'), dpi=150)
plt.close()
print("Saved T5_cnn_activations.png")


## Step 11 — Sample Image Predictions

In [39]:
# ─── Sample Image Predictions ───────────────────────────────────────
np.random.seed(42)
sample_idx = np.random.choice(len(x_test), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
for i, idx in enumerate(sample_idx, start=1):
    ax = axes.flatten()[i - 1]
    ax.imshow(x_test[idx])
    pred_label = CLASS_NAMES[y_pred_c[idx]]
    true_label = CLASS_NAMES[y_true[idx]]
    color = 'green' if y_pred_c[idx] == y_true[idx] else 'red'
    ax.set_title(f'CNN: {pred_label}\nTrue: {true_label}',
                 color=color, fontsize=8)
    ax.axis('off')

plt.suptitle(
    'Custom CNN — CIFAR-10 Sample Predictions (green=correct, red=wrong)',
    fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_sample_predictions.png'), dpi=150)
plt.close()
print("Saved T5_sample_predictions.png")


## Step 12 — Model Comparison

In [40]:
# ─── Model Comparison: Custom CNN vs ResNet-50 ─────────────────────
models_list = ['Custom CNN', 'ResNet-50\n(Transfer Learning)']
test_accs    = [test_acc_c, test_acc_r]
test_losses  = [test_loss_c, test_loss_r]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy
bars = axes[0].bar(models_list, test_accs,
                   color=['#4472C4', '#ED7D31'], edgecolor='black', width=0.5)
axes[0].set_ylim([0, 1.1])
for bar, val in zip(bars, test_accs):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', fontweight='bold')
axes[0].set_title('Test Accuracy Comparison')
axes[0].set_ylabel('Accuracy')
axes[0].grid(axis='y', alpha=0.3)

# Loss
bars2 = axes[1].bar(models_list, test_losses,
                    color=['#4472C4', '#ED7D31'], edgecolor='black', width=0.5)
for bar, val in zip(bars2, test_losses):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.01,
                 f'{val:.4f}', ha='center', fontweight='bold')
axes[1].set_title('Test Loss Comparison')
axes[1].set_ylabel('Loss')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle(
    'Task 5 — CNN Model Comparison on CIFAR-10',
    fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'T5_model_comparison.png'), dpi=150)
plt.close()

print(f"Custom CNN :  acc={test_acc_c:.4f}  loss={test_loss_c:.4f}")
print(f"ResNet-50  :  acc={test_acc_r:.4f}  loss={test_loss_r:.4f}")
print("Saved T5_model_comparison.png")


## Step 13 — Discussion

### Custom CNN Results
| Metric               | Value |
|---|---|
| Test Accuracy        | 0.7256 |
| Test Loss            | 1.0878 |
| Total Parameters     | 654,410 |
| Best Epoch           | 7 / 20 |

The custom CNN reaches approximately 73% test accuracy on CIFAR-10, a reasonable result
for a model trained from scratch in a limited compute environment (no GPU, CPU-only training).
Early stopping restored weights from epoch 7, which prevented overfitting on later epochs
(epochs 14–20 showed train accuracy >0.91 with dropping validation accuracy).

### ResNet-50 Transfer Learning
| Metric               | Value |
|---|---|
| Test Accuracy        | 0.1704 |
| Test Loss            | 2.1552 |
| Trainable Parameters | 529,138 (head only) |

The frozen ResNet-50 achieved only ~17% accuracy — essentially random chance for a 10-class
problem. This confirms that ImageNet pre-trained weights do not transfer well to 32x32
down-sampled CIFAR-10 inputs without fine-tuning. The first Conv2D layer in ResNet-50 was
designed for 224x224 images and the higher-level semantic features learned by later layers
are not useful at such a low spatial resolution.

### Key Takeaways
- **Data augmentation** (rotation, shift, flip, zoom) was configured but training proceeded
  without the augmenter pipeline; adding `datagen.flow(x_train, ...)` during training would
  further improve generalisation.
- **Filter visualisations** (T5_cnn_filters.png) show diverse edge and texture detectors in
  the first convolutional layer — confirming that the model learns structured feature maps.
- **Activation maps** (T5_cnn_activations.png) highlight channel-specific features in early
  convolution blocks, including edge detectors, corner detectors, and colour blobs.
- **Transfer learning caveat**: A frozen ImageNet backbone applied to 32x32 inputs is neither
  operationally nor conceptually correct. Full fine-tuning (unfreeze top 30 layers, LR=1e-5,
  > 20 epochs) is the next step.

### Future Work
1. **ResNet-50 fine-tuning** — unfreeze last 30 layers, LR=1e-5, 20–50 epochs.
2. **Data-augmentation pipeline in training** — use `datagen.flow()` instead of raw `x_train`.
3. **Test-time augmentation** (TTA) to boost final prediction accuracy.
4. **Modern architectures** — EfficientNet-B0, ConvNeXt-Tiny as stronger baselines.
5. **Learning-rate warm-up + cosine decay** schedule for smoother convergence.
